In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTModel
from PIL import Image

/nethome/krangan7/miniconda3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import skimage as ski
from skimage import measure

In [6]:
import exp_matrix as em
import morph_metrics as mm

In [7]:
hallou_data_1 = pd.read_csv('/nethome/krangan7/hallou/dataset1_registered_molecules.csv')
hallou_data_2 = pd.read_csv('/nethome/krangan7/hallou/dataset2_registered_molecules.csv')
hallou_data_3 = pd.read_csv('/nethome/krangan7/hallou/dataset3_registered_molecules.csv')

hallou_data_filtered1 = em.preprocess(hallou_data_1)
hallou_data_filtered2 = em.preprocess(hallou_data_2)
hallou_data_filtered3 = em.preprocess(hallou_data_3)


In [8]:
exp_mask = ski.io.imread('/nethome/krangan7/hallou/segmentation_final.tif')
exp_mask_2 = ski.io.imread('/nethome/krangan7/hallou/segmentation_final_2.tif')
exp_mask_3 = ski.io.imread('/nethome/krangan7/hallou/segmentation_final_3.tif')

In [9]:
scalar_metrics_1 = mm.scalar_morph(exp_mask)
scalar_metrics_2 = mm.scalar_morph(exp_mask_2)
scalar_metrics_3 = mm.scalar_morph(exp_mask_3)


In [10]:
embedded_metrics_1 = mm.embed_mask(exp_mask, batch_size=32)
embedded_metrics_2 = mm.embed_mask(exp_mask_2, batch_size=32)
embedded_metrics_3 = mm.embed_mask(exp_mask_3, batch_size=32)


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 4256.65it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3776.04it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3834.46it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key               | Status     |  | 
--------

In [11]:
tensionmap_data_1 = pd.read_csv('/nethome/krangan7/hallou/TensionMap-main/reproduce_data/dataset1/tensionmap_res.csv')
tensionmap_data_2 = pd.read_csv('/nethome/krangan7/hallou/tensionmap_res_2.csv')
tensionmap_data_3 = pd.read_csv('/nethome/krangan7/hallou/tensionmap_res_3.csv')


In [12]:
def expression_data(hallou_data_filtered):
    exp_matrix = em.create_expression_matrix(hallou_data_filtered)
    expression_pcs = em.expression_pca(exp_matrix, n_components=20)
    expression_pcs = expression_pcs.add_prefix('expr_')  
    return expression_pcs,exp_matrix

In [13]:
def morph_data(embeddings, scalar_metrics):
    pcs_embeddings = mm.pca_embeddings(embeddings)
    scalar_metrics = scalar_metrics.add_prefix('morph_')  
    pcs_embeddings = pcs_embeddings.add_prefix('morph_')
    return pd.concat([pcs_embeddings, scalar_metrics], axis=1)

In [14]:
def tension_map(tension_data, id_map):
    tensionmap_data = tension_data[tension_data['celltype'] != 'Low quality']
    tm = tensionmap_data.set_index('Unnamed: 0')

    tm_m = (tm.reset_index().rename(columns={'Unnamed: 0': 'cell_id'})
            .merge(id_map, on='cell_id', how='inner')
            .set_index('mask_label'))
    cx, cy = tm_m['centroid_x'], tm_m['centroid_y']
    tm_m['spatial_block'] = ((cx > cx.median()).astype(int) * 2 + (cy > cy.median()).astype(int))
    tm_m = tm_m.add_prefix('tm_')
    return tm_m



In [15]:
def join(expr_pcs, morph_data, tm_m):
    final_df_morph = expr_pcs.join(morph_data,how='inner').join(tm_m[['tm_spatial_block', 'tm_celltype', 'tm_boundary_annotation', 'tm_cell_id']],how='inner')
    return final_df_morph

Dataset 1

In [16]:
link1 = hallou_data_filtered1[['mask_label', 'target_cell']].drop_duplicates()
id_map1 = (link1.assign(cell_id=link1['target_cell'].astype(str))[['cell_id', 'mask_label']]
              .drop_duplicates())

In [17]:
expression_data1, exp_matrix1 = expression_data(hallou_data_filtered1)
morph_data1 = morph_data(embedded_metrics_1, scalar_metrics_1)
tension_map1 = tension_map(tensionmap_data_1, id_map1)

joined_df1 = join(expression_data1, morph_data1, tension_map1)

dropped 49 cells, kept 1114


Dataset 2

In [18]:
link2 = hallou_data_filtered2[['mask_label', 'target_cell']].drop_duplicates()
id_map2 = (link2.assign(cell_id=link2['target_cell'].astype(str))[['cell_id', 'mask_label']]
              .drop_duplicates())

In [19]:
expression_data2, exp_matrix2 = expression_data(hallou_data_filtered2)
morph_data2 = morph_data(embedded_metrics_2, scalar_metrics_2)
tension_map2 = tension_map(tensionmap_data_2, id_map2)

joined_df2 = join(expression_data2, morph_data2, tension_map2)

dropped 27 cells, kept 884


Dataset 3

In [20]:
link3 = hallou_data_filtered3[['mask_label', 'target_cell']].drop_duplicates()
id_map3 = (link3.assign(cell_id=link3['target_cell'].astype(str))[['cell_id', 'mask_label']]
              .drop_duplicates())

In [21]:
expression_data3, exp_matrix3 = expression_data(hallou_data_filtered3)
morph_data3 = morph_data(embedded_metrics_3, scalar_metrics_3)
tension_map3 = tension_map(tensionmap_data_3, id_map3)

joined_df3 = join(expression_data3, morph_data3, tension_map3)

dropped 25 cells, kept 890


In [22]:
joined_df3

,expr_PC1,expr_PC2,expr_PC3,expr_PC4,expr_PC5,expr_PC6,expr_PC7,expr_PC8,expr_PC9,expr_PC10,...,morph_extent,morph_orientation,morph_major_axis_length,morph_minor_axis_length,morph_circularity,morph_aspect_ratio,tm_spatial_block,tm_celltype,tm_boundary_annotation,tm_cell_id
mask_label,,,,,,,,,,,,,,,,,,,,,
43,9.083095,-2.787637,-3.886999,-1.202194,-1.033470,-2.172033,-2.022468,-0.305434,2.479263,-4.222057,...,0.472625,0.851466,155.458851,66.092426,0.576101,2.352143,2,Neural crest,Hindbrain,cell_5
45,3.358811,-1.206842,-3.353326,-0.492372,-0.491509,2.017663,0.472392,1.428129,-3.395280,0.546640,...,0.583286,0.899400,149.227316,72.468149,0.664430,2.059212,2,Forebrain/Midbrain/Hindbrain,Hindbrain,cell_6
49,6.192540,4.289817,-1.383085,2.993233,-0.705237,1.187349,2.015276,-0.872244,2.862471,0.147781,...,0.362122,-0.571870,216.750148,65.431446,0.383917,3.312630,0,Forebrain/Midbrain/Hindbrain,Midbrain,cell_9
52,1.316728,2.499251,-2.328587,3.271791,-0.180797,-0.551893,1.290274,-4.570782,1.176277,-0.048625,...,0.520955,-0.230862,207.942278,70.692031,0.510741,2.941524,0,Forebrain/Midbrain/Hindbrain,Midbrain,cell_10
54,3.128155,-2.897985,-0.335002,-0.279580,-1.106311,-3.360940,1.835246,-2.018043,-0.586968,-4.553365,...,0.682314,1.517583,149.286200,61.545862,0.606796,2.425609,2,Surface ectoderm,Hindbrain,cell_11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408,-4.312479,-5.766491,0.727461,-0.822614,0.302080,1.697981,3.254060,2.330244,3.199616,0.821145,...,0.585621,1.373381,74.387011,57.514269,0.656333,1.293366,1,Forebrain/Midbrain/Hindbrain,Other,cell_911
1409,-4.634804,-2.087311,2.547177,0.529365,1.566169,1.589451,2.229607,-0.098193,0.660508,-0.229611,...,0.661638,-0.827976,53.387535,47.755549,0.712074,1.117934,1,Forebrain/Midbrain/Hindbrain,Other,cell_912
1410,1.582523,-5.025133,0.224570,2.833497,1.763142,0.723220,2.797369,2.658414,2.187899,1.678293,...,0.544064,1.276692,107.111373,51.915663,0.523711,2.063180,3,Forebrain/Midbrain/Hindbrain,Other,cell_913


In [23]:
dfs = {'ds1': joined_df1, 'ds2': joined_df2, 'ds3': joined_df3}      # each already one row per mask_label

parts = []
for name, d in dfs.items():
    d = d.copy()
    d['dataset_id'] = name
    d['mask_label'] = d.index                    # keep the original label as a column
    d.index = name + '_' + d.index.astype(str)   # e.g. ds1_12
    parts.append(d)

final_all = pd.concat(parts, axis=0, join='inner')   # 'inner' keeps only columns present in all three


In [54]:
final_all.to_csv('/nethome/krangan7/hallou/final_data_morph_gene.csv', index=True)

In [24]:
exp_matrix1.to_csv('/nethome/krangan7/hallou/exp_matrix1.csv', index=True)
exp_matrix2.to_csv('/nethome/krangan7/hallou/exp_matrix2.csv', index=True)
exp_matrix3.to_csv('/nethome/krangan7/hallou/exp_matrix3.csv', index=True)
